# aither-kvcache v2.0 — Quick Start

Two compression engines for LLM KV caches:

| Engine | Compression | Method | Calibration |
|--------|------------|--------|-------------|
| **TurboQuant** | 3.8–7.1× vs FP16 | Vector quantization (2–4 bit) | None needed |
| **TriAttention** *(NEW)* | ~10× vs FP16 | Spectral (top RoPE frequencies) | Per-model profiles |

This notebook covers:
1. Install & verify
2. TurboQuant standalone
3. TriAttention standalone
4. **vLLM integration** — one flag, no code changes
5. Memory calculator
6. Graph-aware eviction

**Requirements:** Python 3.10+, PyTorch 2.0+, CUDA GPU (Ampere or newer)

## 1. Installation

In [ ]:
!pip install -q "aither-kvcache>=2.0.1"

In [ ]:
import torch
import aither_kvcache

print(f"aither-kvcache v{aither_kvcache.__version__}")
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM: {vram:.1f} GB")

## 2. TurboQuant — Vector Quantization

Random rotation → Lloyd-Max scalar quantization → bit packing.
No calibration data needed. Works on streaming tokens.

In [ ]:
from aither_kvcache import TurboQuant

device = "cuda" if torch.cuda.is_available() else "cpu"
tq = TurboQuant(head_dim=128, bits=4, device=device)

result = tq.validate(num_vectors=50000)

print(f"MSE:           {result['mse']:.6f}")
print(f"Theory bounds: [{result['mse_theory_lower']:.6f}, {result['mse_theory_upper']:.6f}]")
print(f"Ratio to LB:   {result['mse_ratio_to_lower']:.2f}x (paper claims <= 2.7x)")
print(f"Compression:   {result['compression_vs_fp16']:.1f}x vs FP16, {result['compression_vs_fp8']:.1f}x vs FP8")
print(f"Engine:        {'Triton' if result['triton_active'] else 'PyTorch'}")

In [ ]:
# Encode/decode round-trip
kv = torch.randn(32, 8, 128, device=device, dtype=torch.float16)

packed, norms = tq.encode(kv)
decoded = tq.decode(packed, norms)

mse = ((kv.float() - decoded.float()) ** 2).mean().item()
print(f"Input:  {kv.shape} ({kv.dtype})")
print(f"Packed: {packed.shape} ({packed.dtype}) + norms {norms.shape}")
print(f"Output: {decoded.shape} ({decoded.dtype})")
print(f"MSE:    {mse:.6f}")

## 3. TriAttention — Spectral KV Compression *(NEW in v2.0)*

RoPE attention is a trigonometric series. Instead of storing full K/V vectors,
store just the top-F frequency pair coefficients and quantize to int4.

**26 bytes/token** vs 256 FP16 = **~10x compression**.

In [ ]:
from aither_kvcache.triattention import (
    TriAttention, TriAttentionConfig,
    get_profile, get_config_for_model, ALL_PROFILES,
)

print("Calibrated model profiles:")
for name, p in ALL_PROFILES.items():
    print(f"  {name:.<40s} {p.num_layers}L, {p.num_query_heads}Q/{p.num_kv_heads}KV, {p.model_family}")

In [ ]:
config = get_config_for_model("Nemotron-Orchestrator-8B", coeff_bits=4)
print(config.summary())

# Also works with aliases:
#   get_config_for_model("deepseek-r1:14b")
#   get_config_for_model("meta-llama/Llama-3.1-8B-Instruct")
#   get_config_for_model("Qwen3.5-35B")

## 4. vLLM Integration

### The simple way — one flag

With the [TQ-patched vLLM fork](https://github.com/Aitherium/aitherkvcache),
TurboQuant is a **native KV cache dtype**. No hooks, no patches, no env vars:

```bash
# That's it. One flag.
vllm serve your-model --kv-cache-dtype tq-t4nc
```

| Dtype | Bits | Compression | Notes |
|-------|------|------------|-------|
| `tq-t4nc` | 4 | 3.8x | **Recommended** |
| `tq-t3nc` | 3 | 4.9x | More aggressive |
| `tq-t35nc` | 3.5 | 4.4x | Hybrid |
| `tq-t2nc` | 2 | 7.1x | Maximum compression |

### Production results (RTX 5090, Nemotron-8B-AWQ)

| Metric | Value |
|--------|-------|
| KV cache tokens | **250,928** (19.1 GiB) |
| Max concurrency (40K ctx) | 6.1x simultaneous |
| CUDA graphs | 10 piecewise, 6s capture |
| Model load | 6.4 GiB, 15s |

## 5. Memory Calculator

In [ ]:
models = [
    ("Nemotron-Orch-8B",  36, 8, 128),
    ("DeepSeek-R1-14B",   48, 8, 128),
    ("Llama 3.1 8B",      32, 8, 128),
    ("Qwen3.5-8B",        36, 8, 128),
    ("Qwen3.5-35B",       64, 8, 128),
    ("Llama 3.1 70B",     80, 8, 128),
]
seq_len = 40960

print(f"KV Cache at {seq_len:,} tokens")
print(f"{'Model':<22} {'FP16':>7} {'FP8':>7} {'TQ4':>7} {'TriAttn':>9}")
print("-" * 58)
for name, layers, kv_heads, dim in models:
    vecs = 2 * layers * kv_heads
    fp16 = vecs * dim * 2 * seq_len / (1024**3)
    fp8  = vecs * dim * 1 * seq_len / (1024**3)
    tq4  = vecs * 68  * seq_len / (1024**3)
    tri  = vecs * 26  * seq_len / (1024**3)
    print(f"{name:<22} {fp16:>5.2f}GB {fp8:>5.2f}GB {tq4:>5.2f}GB {tri:>7.2f}GB")

## 6. Graph-Aware KV Cache Eviction

Standard eviction (LRU/FIFO) treats system prompts the same as throwaway tokens.
`KVCacheGraph` builds a relationship graph for structurally-informed eviction.

In [ ]:
from aither_kvcache import KVCacheGraph, GraphEvictionAdvisor

graph = KVCacheGraph(protected_sources={"system", "tools"})
graph.add_block(0, "system",    importance=0.95, token_range=(0, 16))
graph.add_block(1, "system",    importance=0.90, token_range=(16, 32))
graph.add_block(2, "tools",     importance=0.85, token_range=(32, 48))
graph.add_block(3, "user",      importance=0.60, token_range=(48, 64))
graph.add_block(4, "assistant", importance=0.30, token_range=(64, 80))

graph.on_attention_step([0, 1, 2, 3, 4])
graph.on_temporal_sequence([3, 4])

victims = graph.suggest_eviction(n_blocks=2)
print(f"Eviction candidates: blocks {victims}")
print(f"(System blocks 0,1,2 are protected)")
print(f"Graph: {graph.get_stats()['num_nodes']} nodes, {graph.get_stats()['num_edges']} edges")

## Next Steps

- **Docs**: [GitHub](https://github.com/Aitherium/aitherkvcache)
- **Paper**: [arXiv:2504.19874](https://arxiv.org/abs/2504.19874)
- **Discussions**: [GitHub Discussions](https://github.com/Aitherium/aitherkvcache/discussions)
- **Issues**: [GitHub Issues](https://github.com/Aitherium/aitherkvcache/issues)